In [1]:
from langchain_core.runnables import RunnableLambda, RunnableSequence
import hashlib
import re

c:\Users\sangram.a.mohanty\OneDrive - Accenture\Documents\PYTHON\AI\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
def encrypt_password(password: str) -> str:
    """Encrypt the given password using SHA-256."""
    return hashlib.sha256(password.encode('UTF-8')).hexdigest()

In [3]:
encrypt_password = RunnableLambda(encrypt_password)
print(encrypt_password.invoke("hello123"))
print(encrypt_password.batch(["hello123", "world123"]))

27cc6994fc1c01ce6659c6bddca9b69c4c6a9418065e612c69d110b3f7b11f8a
['27cc6994fc1c01ce6659c6bddca9b69c4c6a9418065e612c69d110b3f7b11f8a', '378c52420ef6bec16af5f40552e573f7a2cae1c9728747f9b83807fd73d415fd']


In [4]:
def download_csv_files():
    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]
    
    def load_all_csv_files(file_list):
        for file in file_list:
            print("Data load completed for file:", file)
    return RunnableLambda(list_all_csv_files) | RunnableLambda(load_all_csv_files)

# A RunnableLambda class is a wrapper applied on top of a function. It is used to apply transformations to inputs
# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.
files = download_csv_files()
print(files.invoke("http://example.com/data"))

Data load completed for file: 1.csv
Data load completed for file: 2.csv
Data load completed for file: 3.csv
None


In [5]:
def load_all_csv_files():
    """
    In the first 
    RunnableLambda(lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]}) 
    we're:
        1. Calling list_all_csv_files(x["url"], "n": x["n"]) to get the file list using only the url from the input dict. "n": x["n"] in the input dict is never used in this function. But it is carried forward for next function call.
        2. That "n": x["n"] is used to call chunkify like chunkify(d["file_list"], d["n"])
           This output dict (with file_list and n) is then passed to the second RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])), which uses both values to create the chunks.
    """
    def list_all_csv_files(url: str):
        return ["1.csv", "2.csv", "3.csv"]
    
    def chunkify(file_list, n):
        result = []
        for file in file_list:
            for i in range(n):
                result.append(file.replace(".csv", f"_part{i}.csv"))
        return result
    
    def load_all_csv_files(file_list):
        for file in file_list:
            print(f"Data load completed for file:{file}")
    return RunnableSequence(
        RunnableLambda(lambda x: {"file_list": list_all_csv_files(x["url"]), "n": x["n"]}),
        RunnableLambda(lambda d: chunkify(d["file_list"], d["n"])),
        RunnableLambda(load_all_csv_files)
    )


# A runnable object can be called inside another function further invoked after initializing it to a variable
# In the above function, all runnables objects are executed in sequence. The output of first runnable is passed as input to the second runnable.
files = load_all_csv_files()
print(files.invoke({"url": "http://example.com/data", "n": 2}))

Data load completed for file:1_part0.csv
Data load completed for file:1_part1.csv
Data load completed for file:2_part0.csv
Data load completed for file:2_part1.csv
Data load completed for file:3_part0.csv
Data load completed for file:3_part1.csv
None


In [22]:
def generate_odd_number(n: int) -> list[int]:
    """Generate a list of odd numbers up to n."""
    return [i for i in range(n) if i % 2 != 0]

def sum_of_odd_numbers(odd_numbers: list[int]) -> int:
    """Calculate the sum of a list of odd numbers."""
    return sum(odd_numbers)

def check_palindrome(s: int) -> bool:
    s = str(s)
    return s == s[::-1]

output = RunnableSequence(
    first = RunnableLambda(generate_odd_number),
    middle = [RunnableLambda(sum_of_odd_numbers)],
    last = RunnableLambda(check_palindrome)
)

# output = RunnableSequence(
#     first = RunnableLambda(generate_odd_number),
#     last = RunnableLambda(sum_of_odd_numbers)
# )

print(output.invoke(22)) # 121  is a palindrome
print(output.invoke(10)) # 25 is not a palindrome

True
False
